# Autoencoder Variacional — Dataset Animals-10

Treinamento de um **VAE convolucional** e de um **VAE Condicional (cVAE)** no dataset [Animals-10](https://www.kaggle.com/datasets/alessiocorrado99/animals10) (~27.000 imagens, 10 classes).

---

## Estrutura do notebook

| Seção | Conteúdo |
|---|---|
| 1 | Configuração e importações |
| 2 | Exploração do dataset |
| 3 | Pipeline de dados |
| 4 | Arquitetura do VAE |
| 4b | Arquitetura do cVAE |
| 5 | Função de perda (ELBO) |
| 6 | Configuração do MLflow |
| 7 | Treinamento do VAE |
| 7b | Carregar checkpoint (alternativa ao treinamento) |
| 8 | Curvas de perda |
| 9 | Geração de imagens |
| 10 | Reconstrução |
| 11 | Interpolação no espaço latente |
| 12 | Avaliação — IS & FID |
| 13 | Treinamento do cVAE |
| 14 | cVAE — Geração condicional |
| 15 | Comparação do espaço latente — VAE vs cVAE |

---

> **Como usar:** execute as células em ordem. Para pular o treinamento e usar um checkpoint salvo, execute a seção **7b** em vez da seção **7**.

## 1. Configuração e importações

Carrega todas as bibliotecas necessárias, define o dispositivo de computação (`MPS` → `CUDA` → `CPU`) e fixa as sementes aleatórias para reprodutibilidade.

In [ ]:
import os
import random

import mlflow
import mlflow.pytorch
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from PIL import Image
from tqdm.notebook import tqdm

# Fixa sementes para reprodutibilidade, permitindo comparar resultados entre execuções
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Seleciona o melhor dispositivo de computação disponível:
# MPS = GPU Apple Silicon | CUDA = GPU NVIDIA | CPU = fallback
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Dispositivo: {DEVICE}")
print(f"PyTorch    : {torch.__version__}")
print(f"MLflow     : {mlflow.__version__}")

## 2. Exploração do dataset

**Animals-10** contém imagens coloridas de 10 classes de animais, originalmente rotuladas em italiano.

O que esta seção faz:
- Conta o número de imagens por classe
- Exibe a distribuição em um gráfico de barras
- Mostra uma imagem representativa de cada classe

> **O que observar:** distribuição balanceada entre classes (~2.000–3.000 imagens cada). Variação intra-classe alta — útil para avaliar a diversidade das amostras geradas.

In [ ]:
# Mapeamento: nome da pasta em italiano → rótulo em inglês
LABEL_MAP = {
    "cane":       "dog",
    "cavallo":    "horse",
    "elefante":   "elephant",
    "farfalla":   "butterfly",
    "gallina":    "chicken",
    "gatto":      "cat",
    "mucca":      "cow",
    "pecora":     "sheep",
    "ragno":      "spider",
    "scoiattolo": "squirrel",
}

DATA_DIR = "../archive/raw-img"

# Conta imagens por classe
class_counts = {}
for class_name in sorted(os.listdir(DATA_DIR)):
    class_path = os.path.join(DATA_DIR, class_name)
    if not os.path.isdir(class_path):
        continue
    files = [f for f in os.listdir(class_path)
             if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    class_counts[class_name] = len(files)

total = sum(class_counts.values())
print(f"Total de imagens: {total}\n")
print(f"{'Classe':<12} {'Inglês':<12} {'Imagens':>8}")
print("-" * 35)
for cls, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"{cls:<12} {LABEL_MAP.get(cls, cls):<12} {count:>8}")

In [ ]:
# Gráfico de barras da distribuição por classe
fig, ax = plt.subplots(figsize=(10, 4))
classes = list(class_counts.keys())
counts  = list(class_counts.values())
labels  = [LABEL_MAP.get(c, c) for c in classes]

bars = ax.bar(labels, counts, color="steelblue", edgecolor="white")
ax.set_title("Quantidade de imagens por classe", fontsize=13)
ax.set_xlabel("Classe")
ax.set_ylabel("Número de imagens")

for bar, v in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            str(v), ha="center", va="bottom", fontsize=9)

plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Exibe uma imagem aleatória por classe
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle("Uma amostra por classe", fontsize=13)

for ax, (cls, label) in zip(axes.flat, LABEL_MAP.items()):
    class_path = os.path.join(DATA_DIR, cls)
    files = [f for f in os.listdir(class_path)
             if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    img_path = os.path.join(class_path, random.choice(files))
    img = Image.open(img_path).convert("RGB")
    ax.imshow(img)
    ax.set_title(label, fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 3. Pipeline de dados

Define dois pipelines de carregamento:

| Pipeline | Augmentation | Uso |
|---|---|---|
| `AnimalsDataset` | Não | Treinamento do VAE (sem rótulos) |
| `LabeledAnimalsDataset` | Sim (flip + color jitter) | Treinamento do cVAE (com rótulos) |

**Transformações aplicadas:**
- **Resize** → 64×64 px
- **RandomHorizontalFlip + ColorJitter** (apenas no pipeline rotulado)
- **ToTensor** → float [0, 1]
- **Normalize** → [-1, 1] (compatível com a ativação `Tanh` do decoder)

In [ ]:
IMG_SIZE = 64  # resolução espacial (quadrada)

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),                   # [0, 1]
    transforms.Normalize([0.5]*3, [0.5]*3),  # -> [-1, 1]
])


class AnimalsDataset(Dataset):
    """
    Dataset de imagens Animals-10.

    Percorre `root_dir` e coleta todos os arquivos .jpg/.jpeg/.png encontrados
    em subdiretórios de primeiro nível (um subdiretório por classe).

    Args:
        root_dir  : Caminho para a pasta raiz do dataset.
        transform : Pipeline de transformações torchvision.
    """

    def __init__(self, root_dir: str, transform=transform):
        self.transform = transform
        self.samples: list[str] = []

        for class_name in os.listdir(root_dir):
            class_path = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_path):
                continue
            for fname in os.listdir(class_path):
                if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    self.samples.append(os.path.join(class_path, fname))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> torch.Tensor:
        img = Image.open(self.samples[idx]).convert("RGB")
        return self.transform(img)


BATCH_SIZE = 64

dataset = AnimalsDataset(DATA_DIR)
loader  = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,    # 0 = carrega no processo principal (necessário quando a classe é definida no notebook)
    pin_memory=False,
)

print(f"Total de imagens  : {len(dataset)}")
print(f"Batches por época : {len(loader)}")
print(f"Shape do batch    : {next(iter(loader)).shape}")

# Mapeamentos de classe usados pelo cVAE
CLASS_TO_IDX: dict[str, int] = {cls: i for i, cls in enumerate(sorted(LABEL_MAP))}
IDX_TO_CLASS: dict[int, str] = {i: LABEL_MAP[cls] for cls, i in CLASS_TO_IDX.items()}

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])


class LabeledAnimalsDataset(Dataset):
    """Versão rotulada do dataset — retorna (tensor, índice_de_classe) para o cVAE."""

    def __init__(self, root_dir: str, transform=train_transform):
        self.transform = transform
        self.samples: list[tuple[str, int]] = []
        for class_name in os.listdir(root_dir):
            if class_name not in CLASS_TO_IDX:
                continue
            class_path = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_path):
                continue
            label = CLASS_TO_IDX[class_name]
            for fname in os.listdir(class_path):
                if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                    self.samples.append((os.path.join(class_path, fname), label))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label


def get_dataloader(root_dir: str, batch_size: int = 64, augment: bool = True) -> DataLoader:
    t = train_transform if augment else eval_transform
    ds = LabeledAnimalsDataset(root_dir, transform=t)
    return DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=False)


## 4. Arquitetura do VAE

Um **Autoencoder Variacional** aprende uma distribuição sobre o espaço latente, não apenas uma representação pontual.

```
Imagem x ──► Encoder ──► μ, log σ² ──► z = μ + σ·ε ──► Decoder ──► x̂
                          (posterior         (reparametrização:
                           q(z|x))            ε ~ N(0, I))
```

| Componente | Detalhes |
|---|---|
| **Encoder** | 4× Conv2d (stride 2) + Norm + LeakyReLU → μ, log σ² ∈ ℝ¹²⁸ |
| **Reparametrização** | z = μ + σ·ε — mantém gradientes fluindo pelo encoder |
| **Decoder** | FC + 4× ConvTranspose2d + Norm + ReLU → Tanh → x̂ ∈ [-1,1] |
| **Normalização** | `BatchNorm2d` (padrão) ou `GroupNorm` — configurado via `NORM` |

> **Por que reparametrizar?** Amostrar `z` diretamente quebraria o fluxo de gradientes. Ao separar a estocasticidade em `ε`, μ e σ permanecem diferenciáveis.

In [ ]:
LATENT_DIM  = 128   # dimensionalidade do espaço latente
GROUP_SIZE  = 8     # canais por grupo quando norm="group"


def _norm(norm: str, num_channels: int) -> nn.Module:
    """Retorna BatchNorm2d ou GroupNorm conforme o valor de 'norm'."""
    if norm == "group":
        return nn.GroupNorm(num_channels // GROUP_SIZE, num_channels)
    return nn.BatchNorm2d(num_channels)


class Encoder(nn.Module):
    """
    Encoder convolucional: mapeia uma imagem 3×64×64 para os parâmetros
    (μ, log σ²) de uma distribuição Gaussiana no espaço latente.

    Cada bloco conv usa stride=2 para reduzir a resolução espacial à metade
    (downsampling aprendível, equivalente a Conv + MaxPool).
    Dropout2d descarta mapas de features inteiros para regularizar features espaciais.
    """

    def __init__(self, latent_dim: int = LATENT_DIM, norm: str = "batch"):
        super().__init__()
        self.conv = nn.Sequential(
            # 3 × 64 × 64  ->  32 × 32 × 32
            nn.Conv2d(3,   32,  4, stride=2, padding=1),
            _norm(norm, 32),
            nn.LeakyReLU(0.2),

            # 32 × 32 × 32  ->  64 × 16 × 16
            nn.Conv2d(32,  64,  4, stride=2, padding=1),
            _norm(norm, 64),
            nn.LeakyReLU(0.2),

            # 64 × 16 × 16  ->  128 × 8 × 8
            nn.Conv2d(64,  128, 4, stride=2, padding=1),
            _norm(norm, 128),
            nn.LeakyReLU(0.2),

            # 128 × 8 × 8  ->  256 × 4 × 4
            nn.Conv2d(128, 256, 4, stride=2, padding=1),
            _norm(norm, 256),
            nn.LeakyReLU(0.2),
        )
        flat = 256 * 4 * 4
        self.fc_mu     = nn.Linear(flat, latent_dim)
        self.fc_logvar = nn.Linear(flat, latent_dim)

    def forward(self, x: torch.Tensor):
        h = self.conv(x).view(x.size(0), -1)  # achatamento: (B, 256*4*4)
        return self.fc_mu(h), self.fc_logvar(h)


class Decoder(nn.Module):
    """
    Decoder convolucional: mapeia um vetor latente z ∈ ℝ^{latent_dim}
    de volta para uma imagem 3×64×64 no intervalo [-1, 1].

    ConvTranspose2d com stride=2 dobra a resolução espacial a cada
    passo — imagem espelhada do encoder.
    """

    def __init__(self, latent_dim: int = LATENT_DIM, norm: str = "batch"):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 256 * 4 * 4)
        self.deconv = nn.Sequential(
            # 256 × 4 × 4  ->  128 × 8 × 8
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            _norm(norm, 128),
            nn.ReLU(),

            # 128 × 8 × 8  ->  64 × 16 × 16
            nn.ConvTranspose2d(128, 64,  4, stride=2, padding=1),
            _norm(norm, 64),
            nn.ReLU(),

            # 64 × 16 × 16  ->  32 × 32 × 32
            nn.ConvTranspose2d(64,  32,  4, stride=2, padding=1),
            _norm(norm, 32),
            nn.ReLU(),

            # 32 × 32 × 32  ->  3 × 64 × 64
            nn.ConvTranspose2d(32,  3,   4, stride=2, padding=1),
            nn.Tanh(),  # saída em [-1, 1], correspondendo à normalização dos dados
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        h = self.fc(z).view(z.size(0), 256, 4, 4)  # reshape: (B, 256, 4, 4)
        return self.deconv(h)


class VAE(nn.Module):
    """
    Autoencoder Variacional completo.

    'reparameterize' implementa o truque da reparametrização:
        z = μ + σ·ε,  ε ~ N(0, I)
    Isso mantém o fluxo de gradientes pelo encoder durante a retropropagação,
    pois ε é amostrado independentemente dos parâmetros do modelo.
    No modo eval, retorna μ diretamente (sem ruído).
    """

    def __init__(self, latent_dim: int = LATENT_DIM, norm: str = "batch"):
        super().__init__()
        self.encoder = Encoder(latent_dim, norm)
        self.decoder = Decoder(latent_dim, norm)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        if self.training:
            std = torch.exp(0.5 * logvar)  # σ = exp(0.5 · log σ²)
            eps = torch.randn_like(std)     # ε ~ N(0, I)
            return mu + eps * std
        return mu

    def forward(self, x: torch.Tensor):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar

    @torch.no_grad()
    def generate(self, n: int) -> torch.Tensor:
        """Gera n imagens amostrando z ~ N(0, I) diretamente."""
        z = torch.randn(n, self.encoder.fc_mu.out_features).to(
            next(self.parameters()).device
        )
        return self.decoder(z)



DROPOUT     = 0.2
NUM_CLASSES = 10
EMBED_DIM   = 64


class ConditionalEncoder(nn.Module):
    """Encoder condicional: mapeia (imagem, rótulo de classe) → (μ, log σ²)."""

    def __init__(
        self,
        latent_dim: int = LATENT_DIM,
        num_classes: int = NUM_CLASSES,
        embed_dim: int = EMBED_DIM,
        dropout: float = DROPOUT,
        norm: str = "batch",
    ):
        super().__init__()
        self.embedding = nn.Embedding(num_classes, embed_dim)
        self.conv = nn.Sequential(
            nn.Conv2d(3,   32,  4, stride=2, padding=1), _norm(norm, 32),  nn.LeakyReLU(0.2),
            nn.Conv2d(32,  64,  4, stride=2, padding=1), _norm(norm, 64),  nn.LeakyReLU(0.2), nn.Dropout2d(dropout),
            nn.Conv2d(64,  128, 4, stride=2, padding=1), _norm(norm, 128), nn.LeakyReLU(0.2), nn.Dropout2d(dropout),
            nn.Conv2d(128, 256, 4, stride=2, padding=1), _norm(norm, 256), nn.LeakyReLU(0.2),
        )
        flat = 256 * 4 * 4
        self.pre_latent = nn.Dropout(dropout)
        self.fc_mu      = nn.Linear(flat + embed_dim, latent_dim)
        self.fc_logvar  = nn.Linear(flat + embed_dim, latent_dim)

    def forward(self, x: torch.Tensor, y: torch.Tensor):
        h = self.conv(x).view(x.size(0), -1)
        h = self.pre_latent(h)
        c = self.embedding(y)
        h = torch.cat([h, c], dim=1)
        return self.fc_mu(h), self.fc_logvar(h)


class ConditionalDecoder(nn.Module):
    """Decoder condicional: mapeia (z, rótulo de classe) → imagem 3×64×64."""

    def __init__(
        self,
        latent_dim: int = LATENT_DIM,
        num_classes: int = NUM_CLASSES,
        embed_dim: int = EMBED_DIM,
        dropout: float = DROPOUT,
        norm: str = "batch",
    ):
        super().__init__()
        self.embedding = nn.Embedding(num_classes, embed_dim)
        self.fc = nn.Sequential(
            nn.Linear(latent_dim + embed_dim, 256 * 4 * 4),
            nn.Dropout(dropout),
        )
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), _norm(norm, 128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64,  4, stride=2, padding=1), _norm(norm, 64),  nn.ReLU(), nn.Dropout2d(dropout),
            nn.ConvTranspose2d(64,  32,  4, stride=2, padding=1), _norm(norm, 32),  nn.ReLU(),
            nn.ConvTranspose2d(32,  3,   4, stride=2, padding=1), nn.Tanh(),
        )

    def forward(self, z: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        c = self.embedding(y)
        h = self.fc(torch.cat([z, c], dim=1))
        return self.deconv(h.view(z.size(0), 256, 4, 4))


class CVAE(nn.Module):
    """VAE Condicional: encoder e decoder condicionados ao rótulo de classe y."""

    def __init__(
        self,
        latent_dim: int = LATENT_DIM,
        num_classes: int = NUM_CLASSES,
        embed_dim: int = EMBED_DIM,
        dropout: float = DROPOUT,
        norm: str = "batch",
    ):
        super().__init__()
        self.encoder = ConditionalEncoder(latent_dim, num_classes, embed_dim, dropout, norm)
        self.decoder = ConditionalDecoder(latent_dim, num_classes, embed_dim, dropout, norm)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        if self.training:
            return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return mu

    def forward(self, x: torch.Tensor, y: torch.Tensor):
        mu, logvar = self.encoder(x, y)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z, y), mu, logvar

    @torch.no_grad()
    def generate(self, y: torch.Tensor) -> torch.Tensor:
        z = torch.randn(y.size(0), self.encoder.fc_mu.out_features).to(y.device)
        return self.decoder(z, y)


def get_beta(epoch: int, warmup: int = 25, beta_max: float = 1.0) -> float:
    return min(epoch / warmup, 1.0) * beta_max


model = VAE(latent_dim=LATENT_DIM).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total de parâmetros     : {total_params:,}")
print(f"Parâmetros treináveis   : {train_params:,}")
print(model)

## 4b. Arquitetura do cVAE

O **VAE Condicional** estende o VAE injetando o rótulo de classe `y` tanto no encoder quanto no decoder via um embedding aprendido:

```
Imagem x ──► ConditionalEncoder ──► μ, log σ² ──► z = μ + σ·ε ──► ConditionalDecoder ──► x̂
                     ↑                                                       ↑
                 embed(y)                                               embed(y)
```

| Onde | Condicionamento |
|---|---|
| **Encoder** | `cat([conv_features, embed(y)])` → cabeças μ / log σ² |
| **Decoder** | `cat([z, embed(y)])` → projeção FC |

- **Embedding:** `nn.Embedding(10, 64)` — projeta cada classe em um vetor de 64 dimensões  
- **Inferência:** `CVAE.generate(y)` amostra `z ~ N(0, I)` e decodifica com a classe alvo

> **Resultado esperado:** o cVAE deve aprender regiões distintas por classe no espaço latente, ao contrário do VAE não-condicional.

In [ ]:
cvae = CVAE(latent_dim=LATENT_DIM).to(DEVICE)

total_params = sum(p.numel() for p in cvae.parameters())
train_params = sum(p.numel() for p in cvae.parameters() if p.requires_grad)
print(f"NUM_CLASSES             : {NUM_CLASSES}")
print(f"EMBED_DIM               : {EMBED_DIM}")
print(f"Total de parâmetros     : {total_params:,}")
print(f"Parâmetros treináveis   : {train_params:,}")
print()
print(cvae)

## 5. Função de perda — ELBO

O VAE minimiza o negativo do **Evidence Lower Bound (ELBO)**:

$$\mathcal{L} = \underbrace{\mathbb{E}[\log p(x|z)]}_{\text{reconstrução (MSE)}} - \beta \cdot \underbrace{D_{KL}(q(z|x) \| p(z))}_{\text{regularização KL}}$$

| Termo | Papel |
|---|---|
| **Reconstrução (MSE)** | Penaliza diferenças pixel a pixel entre `x` e `x̂` |
| **Divergência KL** | Força `q(z|x)` em direção à priori `N(0, I)` — regulariza o espaço latente |
| **β (annealing)** | Sobe de 0 → 1 ao longo de `KL_WARMUP` épocas, evitando colapso do posterior no início |

**Forma fechada da KL entre Gaussianas:**
$$D_{KL} = -\frac{1}{2} \sum_{j=1}^{d} \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

> **Sinal de colapso KL:** se a KL cair para ~0 durante o treinamento, o encoder parou de usar a entrada — o decoder passa a gerar apenas a média do dataset.

In [ ]:
def vae_loss(
    recon: torch.Tensor,
    x: torch.Tensor,
    mu: torch.Tensor,
    logvar: torch.Tensor,
    beta: float = 1.0,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Calcula a função de perda ELBO do VAE.

    Args:
        recon   : Imagem reconstruída pelo decoder, shape (B, C, H, W).
        x       : Imagem original, shape (B, C, H, W).
        mu      : Médias da distribuição latente, shape (B, latent_dim).
        logvar  : Log-variâncias da distribuição latente, shape (B, latent_dim).
        beta    : Peso da KL (padrão=1). Aumentar para comportamento β-VAE.

    Returns:
        total_loss : Perda escalar combinada.
        recon_loss : Termo de reconstrução (MSE por amostra).
        kl_loss    : Termo de divergência KL por amostra.
    """
    # MSE somado sobre pixels, com média sobre o batch
    recon_loss = F.mse_loss(recon, x, reduction="sum") / x.size(0)

    # KL em forma fechada para Gaussianas: somado sobre dims latentes, média sobre batch
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)

    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss, kl_loss

## 6. Configuração do experimento MLflow

O **MLflow** rastreia cada execução automaticamente, registrando:

| O que | Exemplos |
|---|---|
| **Parâmetros** | épocas, lr, β, `latent_dim`, `batch_size` |
| **Métricas por época** | `loss_total`, `loss_recon`, `loss_kl`, `beta`, `lr` |
| **Artefatos** | checkpoints `.pt`, imagens geradas `.png`, modelo final |

Cada execução da célula de treinamento cria uma nova entrada no MLflow, facilitando a comparação entre configurações.

**Para abrir a interface:**
```bash
source venv/bin/activate
mlflow ui
```
Acesse `http://localhost:5000`.

In [ ]:
# Todas as execuções são agrupadas sob este nome de experimento
EXPERIMENT_NAME = "cvae-animals10"

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Experimento MLflow: '{EXPERIMENT_NAME}'")
print(f"URI de rastreamento: {mlflow.get_tracking_uri()}")

## 7. Treinamento do VAE

> **Atenção:** esta célula treina do zero (~50 épocas). Se já existe um checkpoint salvo, use a **seção 7b** abaixo para carregá-lo diretamente.

**Hiperparâmetros configuráveis:**

| Parâmetro | Valor padrão | Descrição |
|---|---|---|
| `EPOCHS` | 50 | Passagens completas sobre o dataset |
| `LR` | 1e-3 | Taxa de aprendizado inicial (Adam) |
| `BETA` | 1.0 | Peso da KL — 1.0 = VAE padrão |
| `NORM` | `"batch"` | Normalização: `"batch"` ou `"group"` |
| `LATENT_DIM` | 128 | Dimensões do espaço latente |
| `BATCH_SIZE` | 64 | Amostras por passo de gradiente |

In [ ]:
# ── Hiperparâmetros ───────────────────────────────────────────────────────────
EPOCHS     = 50
LR         = 1e-3
BETA       = 1.0
NORM       = "batch"  # "batch" | "group"
SAVE_EVERY = 5
CKPT_DIR   = "checkpoints"
OUTPUT_DIR = "../outputs"
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(CKPT_DIR, exist_ok=True)

# Reinstancia o modelo para que cada execução comece do zero
model     = VAE(latent_dim=LATENT_DIM, norm=NORM).to(DEVICE)
optimizer = Adam(model.parameters(), lr=LR)

# Reduz o LR pela metade se a perda não melhorar por 3 épocas consecutivas
scheduler = ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

history = {"total": [], "recon": [], "kl": [], "beta": []}

with mlflow.start_run() as run:
    print(f"MLflow run ID: {run.info.run_id}")

    # Registra todos os hiperparâmetros para que cada execução seja totalmente reproduzível
    mlflow.log_params({
        "epochs":      EPOCHS,
        "lr":          LR,
        "beta":        BETA,
        "latent_dim":  LATENT_DIM,
        "img_size":    IMG_SIZE,
        "batch_size":  BATCH_SIZE,
        "optimizer":   "Adam",
        "device":      str(DEVICE),
        "seed":        SEED,
    })

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_sum = recon_sum = kl_sum = 0.0

        for batch in tqdm(loader, desc=f"Época {epoch}/{EPOCHS}", leave=False):
            x = batch.to(DEVICE)

            # Passagem direta
            recon, mu, logvar = model(x)
            loss, recon_loss, kl_loss = vae_loss(recon, x, mu, logvar, beta=BETA)

            # Retropropagação
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_sum += loss.item()
            recon_sum += recon_loss.item()
            kl_sum    += kl_loss.item()

        # Médias por batch
        n = len(loader)
        avg_total = total_sum / n
        avg_recon = recon_sum / n
        avg_kl    = kl_sum    / n

        history["total"].append(avg_total)
        history["recon"].append(avg_recon)
        history["kl"].append(avg_kl)
        history["beta"].append(BETA)

        scheduler.step(avg_total)

        # Registra métricas no MLflow — um ponto de dados por época
        mlflow.log_metrics({
            "loss_total": avg_total,
            "loss_recon": avg_recon,
            "loss_kl":    avg_kl,
            "beta":       BETA,
            "lr":         optimizer.param_groups[0]["lr"],
        }, step=epoch)

        print(f"Época {epoch:3d}/{EPOCHS} | "
              f"loss={avg_total:8.2f}  "
              f"recon={avg_recon:8.2f}  "
              f"kl={avg_kl:6.2f}")

        if epoch % SAVE_EVERY == 0:
            ckpt_path = os.path.join(CKPT_DIR, f"vae_epoch{epoch:03d}.pt")
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "history": history,
            }, ckpt_path)
            mlflow.log_artifact(ckpt_path, artifact_path="checkpoints")
            print(f"  Checkpoint salvo e registrado: {ckpt_path}")

    # Salva e registra o modelo final
    final_path = os.path.join(CKPT_DIR, "vae_final.pt")
    torch.save({"epoch": EPOCHS, "model_state": model.state_dict(), "history": history},
               final_path)
    mlflow.pytorch.log_model(model, artifact_path="model")
    mlflow.log_artifact(final_path, artifact_path="checkpoints")

    print(f"\nTreinamento concluído. Run ID: {run.info.run_id}")
    print(f"Ver resultados: mlflow ui  (http://localhost:5000)")

## 7b. Carregar checkpoint

> Use esta seção **em vez da seção 7** quando o modelo já foi treinado e você quer apenas explorar os resultados.

O checkpoint `vae_final.pt` contém os pesos do modelo e o histórico de perda completo — todas as células de visualização a seguir funcionarão normalmente.

In [ ]:
# ── Configuração (deve estar alinhada com a execução original) ────────────
NORM       = "batch"      # "batch" | "group"
CKPT_DIR   = "checkpoints"
OUTPUT_DIR = "../outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
# ─────────────────────────────────────────────────────────────────────────

ckpt = torch.load(os.path.join(CKPT_DIR, "vae_final.pt"), map_location=DEVICE, weights_only=False)

model = VAE(latent_dim=LATENT_DIM, norm=NORM).to(DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()

history = ckpt.get("history", {"total": [], "recon": [], "kl": [], "beta": []})

print(f"Checkpoint carregado — época {ckpt['epoch']}")
print(f"Dispositivo: {DEVICE}")

## 8. Curvas de perda

Visualiza a evolução das métricas ao longo das épocas.

**O que observar:**

| Curva | Comportamento saudável | Sinal de problema |
|---|---|---|
| **Perda total** | Queda suave e contínua | Plateau precoce ou divergência |
| **Reconstrução (MSE)** | Queda rápida nas primeiras épocas | Permanece alta → encoder não aprende |
| **KL** | Sobe gradualmente, estabiliza ~150 | Colapsa para 0 → posterior inativo |
| **β** | Rampa linear 0 → 1 em `KL_WARMUP` épocas | — |

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
panels = [
    ("total", "Perda total (ELBO)",          "royalblue"),
    ("recon", "Perda de reconstrução (MSE)", "darkorange"),
    ("kl",    "Divergência KL",              "seagreen"),
    ("beta",  "Escalonamento de β",          "mediumpurple"),
]

for ax, (key, title, color) in zip(axes, panels):
    data   = history.get(key, [])
    epochs = range(1, len(data) + 1)
    ax.plot(epochs, data, color=color, linewidth=2)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Época")
    ax.set_ylabel("Perda")
    ax.grid(alpha=0.3)

plt.suptitle("Histórico de treinamento do VAE", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_history.png"), dpi=150, bbox_inches="tight")
plt.show()

# Registra o gráfico como artefato para que apareça na execução do MLflow
mlflow.log_artifact(os.path.join(OUTPUT_DIR, "training_history.png"))
_path = os.path.join(OUTPUT_DIR, "training_history.png")
print(f"Salvo em {_path}")

## 9. Geração de imagens

Gera amostras inéditas amostrando `z ~ N(0, I)` diretamente — **sem usar o encoder**.

> **O que observar:** diversidade e coerência visual das amostras. Imagens borradas e sem estrutura indicam colapso de KL. Variedade de formas e cores indica que o espaço latente está bem distribuído.

In [ ]:
def denormalize(tensor: torch.Tensor) -> torch.Tensor:
    """Reverte a normalização [-1, 1] para [0, 1] para exibição."""
    return (tensor * 0.5 + 0.5).clamp(0, 1)


def show_grid(imgs: torch.Tensor, title: str, cols: int = 8, save_as: str = None):
    """
    Exibe uma grade de imagens.

    Args:
        imgs    : Tensor (N, C, H, W) no intervalo [0, 1].
        title   : Título do gráfico.
        cols    : Número de colunas na grade.
        save_as : Caminho para salvar a figura (opcional).
    """
    n = imgs.shape[0]
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 1.6))

    for i, ax in enumerate(axes.flat):
        if i < n:
            ax.imshow(imgs[i].permute(1, 2, 0).cpu().numpy())
        ax.axis("off")

    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    if save_as:
        plt.savefig(save_as, dpi=150, bbox_inches="tight")
        mlflow.log_artifact(save_as)
        print(f"Salvo em {save_as}")
    plt.show()


# Gera 32 imagens inéditas
model.eval()
generated = denormalize(model.generate(32))
show_grid(generated, "Amostras geradas pelo VAE (z ~ N(0, I))",
          cols=8, save_as=os.path.join(OUTPUT_DIR, "generated_samples.png"))

## 10. Reconstrução

Passa imagens reais pelo ciclo completo `encoder → decoder` e compara com os originais.

> **O que observar:** as colunas ímpares são originais; as pares são reconstruções. Alguma perda de detalhe fino (pelos, bordas) é esperada com MSE — o modelo tende a gerar versões "suavizadas".

In [ ]:
model.eval()
with torch.no_grad():
    sample_batch = next(iter(loader))[:8].to(DEVICE)
    recon_batch, _, _ = model(sample_batch)

originals     = denormalize(sample_batch)
reconstructed = denormalize(recon_batch)

# Intercala originais e reconstruções na mesma linha
interleaved = torch.stack(
    [img for pair in zip(originals, reconstructed) for img in pair]
)

show_grid(
    interleaved,
    "Original (colunas ímpares) vs. Reconstruído (colunas pares)",
    cols=8,
    save_as=os.path.join(OUTPUT_DIR, "reconstructions.png"),
)

## 11. Interpolação no espaço latente

Uma das propriedades mais importantes de um VAE é que seu espaço latente é **contínuo e estruturado** — pontos próximos produzem imagens visualmente similares.

Interpolação linear entre dois vetores latentes:

$$z(\alpha) = (1-\alpha)\, z_1 + \alpha\, z_2, \quad \alpha \in [0, 1]$$

> **O que observar:** a transição deve ser **suave e gradual**. Borrão excessivo no meio (α ≈ 0,5) indica lacunas no espaço latente — região não coberta pelo encoder.

In [ ]:
def interpolate_latent(model: VAE, steps: int = 10) -> None:
    """
    Gera uma sequência de imagens interpolando linearmente entre
    dois pontos aleatórios no espaço latente.

    Args:
        model : Modelo VAE treinado.
        steps : Número de passos de interpolação.
    """
    device = next(model.parameters()).device

    z1 = torch.randn(1, LATENT_DIM, device=device)
    z2 = torch.randn(1, LATENT_DIM, device=device)

    # α varre de 0 a 1 em `steps` pontos igualmente espaçados
    alphas = torch.linspace(0, 1, steps, device=device)
    zs = torch.stack([(1 - a) * z1 + a * z2 for a in alphas]).squeeze(1)

    with torch.no_grad():
        imgs = denormalize(model.decoder(zs))

    fig, axes = plt.subplots(1, steps, figsize=(steps * 1.8, 2))
    for i, ax in enumerate(axes):
        ax.imshow(imgs[i].permute(1, 2, 0).cpu().numpy())
        ax.set_title(f"α={alphas[i].item():.1f}", fontsize=8)
        ax.axis("off")

    plt.suptitle("Interpolação no espaço latente  (z₁ → z₂)", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "interpolation.png"), dpi=150, bbox_inches="tight")
    mlflow.log_artifact(os.path.join(OUTPUT_DIR, "interpolation.png"))
    plt.show()
    _path = os.path.join(OUTPUT_DIR, "interpolation.png")
print(f"Salvo em {_path}")


interpolate_latent(model, steps=10)

## 12. Avaliação — IS & FID

Métricas quantitativas para comparar a qualidade das imagens geradas.

| Métrica | O que mede | Direção |
|---|---|---|
| **Inception Score (IS)** | Qualidade + diversidade — usa predições do Inception-v3 | Maior = melhor |
| **Fréchet Inception Distance (FID)** | Distância entre distribuições real e gerada no espaço de features | Menor = melhor |

> **Nota:** as imagens são redimensionadas de 64×64 → 299×299 para o Inception-v3. Os valores absolutos são menos importantes do que a **diferença relativa entre execuções** — use para comparar VAE vs cVAE.

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore

N_EVAL = 2048  # imagens para avaliação — trade-off entre precisão e velocidade

def to_01(t: torch.Tensor) -> torch.Tensor:
    """Converte tensor [-1, 1] para float [0, 1]."""
    return (t * 0.5 + 0.5).clamp(0, 1)

print(f"Avaliando com {N_EVAL} imagens (Inception-v3 baixa ~100 MB na primeira execução)...\n")

model.eval()

# ── Inception Score (somente imagens geradas) ─────────────────────────────────
is_metric = InceptionScore(normalize=True)
with torch.no_grad():
    for start in range(0, N_EVAL, BATCH_SIZE):
        n = min(BATCH_SIZE, N_EVAL - start)
        z = torch.randn(n, LATENT_DIM, device=DEVICE)
        is_metric.update(to_01(model.decoder(z)).cpu())

is_mean, is_std = is_metric.compute()
print(f"Inception Score : {is_mean:.3f} ± {is_std:.3f}  (maior = melhor)")

# ── Fréchet Inception Distance (real vs gerado) ───────────────────────────────
fid_metric = FrechetInceptionDistance(normalize=True)

real_seen = 0
with torch.no_grad():
    for batch in loader:
        fid_metric.update(to_01(batch).cpu(), real=True)
        real_seen += batch.shape[0]
        if real_seen >= N_EVAL:
            break

with torch.no_grad():
    for start in range(0, N_EVAL, BATCH_SIZE):
        n = min(BATCH_SIZE, N_EVAL - start)
        z = torch.randn(n, LATENT_DIM, device=DEVICE)
        fid_metric.update(to_01(model.decoder(z)).cpu(), real=False)

fid_score = fid_metric.compute()
print(f"FID             : {fid_score:.2f}  (menor = melhor)")

# ── Registra na execução existente do MLflow ──────────────────────────────────
with mlflow.start_run(run_id=run.info.run_id):
    mlflow.log_metrics({"is_mean": float(is_mean), "is_std": float(is_std), "fid": float(fid_score)})
print("\nMétricas registradas no MLflow.")

## 13. Treinamento do cVAE

> **Atenção:** esta célula treina do zero. Se já existe um checkpoint `cvae_final.pt`, adapte a seção 7b para carregá-lo.

Mesma estrutura da seção 7, com duas diferenças:

1. O DataLoader retorna pares `(imagem, rótulo)` — usando `LabeledAnimalsDataset`
2. O modelo recebe o rótulo `y` a cada passagem: `cvae_model(x, y)`

O annealing de KL é aplicado de forma idêntica para uma comparação justa com o VAE.

In [ ]:
# ── Hiperparâmetros ───────────────────────────────────────────────────────────
CVAE_EPOCHS     = 50
CVAE_LR         = 1e-3
CVAE_BETA_MAX   = 1.0
CVAE_KL_WARMUP  = 25
CVAE_SAVE_EVERY = 5
CVAE_NORM       = "batch"  # "batch" | "group"
# ─────────────────────────────────────────────────────────────────────────────

cvae_loader    = get_dataloader(DATA_DIR, batch_size=BATCH_SIZE, augment=True)
cvae_model     = CVAE(latent_dim=LATENT_DIM, num_classes=NUM_CLASSES, embed_dim=EMBED_DIM, norm=CVAE_NORM).to(DEVICE)
cvae_optimizer = Adam(cvae_model.parameters(), lr=CVAE_LR)
cvae_scheduler = ReduceLROnPlateau(cvae_optimizer, patience=3, factor=0.5)

cvae_history = {"total": [], "recon": [], "kl": [], "beta": []}

with mlflow.start_run() as cvae_run:
    print(f"MLflow run ID: {cvae_run.info.run_id}")

    mlflow.log_params({
        "model_type":  "cvae",
        "epochs":      CVAE_EPOCHS,
        "lr":          CVAE_LR,
        "beta_max":    CVAE_BETA_MAX,
        "kl_warmup":   CVAE_KL_WARMUP,
        "latent_dim":  LATENT_DIM,
        "embed_dim":   EMBED_DIM,
        "num_classes": NUM_CLASSES,
        "batch_size":  BATCH_SIZE,
        "optimizer":   "Adam",
        "device":      str(DEVICE),
    })

    for epoch in range(1, CVAE_EPOCHS + 1):
        cvae_model.train()
        total_sum = recon_sum = kl_sum = 0.0
        beta = get_beta(epoch, warmup=CVAE_KL_WARMUP, beta_max=CVAE_BETA_MAX)

        for x, y in tqdm(cvae_loader, desc=f"Época {epoch}/{CVAE_EPOCHS}", leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            cvae_optimizer.zero_grad()
            recon, mu, logvar = cvae_model(x, y)
            loss, recon_loss, kl_loss = vae_loss(recon, x, mu, logvar, beta=beta)
            loss.backward()
            cvae_optimizer.step()

            total_sum += loss.item()
            recon_sum += recon_loss.item()
            kl_sum    += kl_loss.item()

        n = len(cvae_loader)
        avg_total = total_sum / n
        avg_recon = recon_sum / n
        avg_kl    = kl_sum    / n

        cvae_history["total"].append(avg_total)
        cvae_history["recon"].append(avg_recon)
        cvae_history["kl"].append(avg_kl)
        cvae_history["beta"].append(beta)

        cvae_scheduler.step(avg_total)

        mlflow.log_metrics({
            "loss_total": avg_total,
            "loss_recon": avg_recon,
            "loss_kl":    avg_kl,
            "beta":       beta,
            "lr":         cvae_optimizer.param_groups[0]["lr"],
        }, step=epoch)

        print(f"Época {epoch:3d}/{CVAE_EPOCHS} | β={beta:.2f} | "
              f"loss={avg_total:.2f}  recon={avg_recon:.2f}  kl={avg_kl:.2f}")

        if epoch % CVAE_SAVE_EVERY == 0:
            ckpt_path = os.path.join(CKPT_DIR, f"cvae_epoch{epoch:03d}.pt")
            torch.save({
                "epoch": epoch,
                "model_state": cvae_model.state_dict(),
                "optimizer_state": cvae_optimizer.state_dict(),
                "history": cvae_history,
            }, ckpt_path)
            mlflow.log_artifact(ckpt_path, artifact_path="checkpoints")
            print(f"  Checkpoint salvo e registrado: {ckpt_path}")

    cvae_final_path = os.path.join(CKPT_DIR, "cvae_final.pt")
    torch.save({"epoch": CVAE_EPOCHS, "model_state": cvae_model.state_dict(),
                "history": cvae_history}, cvae_final_path)
    mlflow.pytorch.log_model(cvae_model, artifact_path="model")
    mlflow.log_artifact(cvae_final_path, artifact_path="checkpoints")

    print(f"\nTreinamento concluído. Run ID: {cvae_run.info.run_id}")
    print(f"Ver resultados: mlflow ui  →  http://localhost:5000")

## 14. cVAE — Geração condicional

Ao contrário do VAE, o cVAE recebe um rótulo de classe `y` e gera imagens daquela classe específica:

```
z ~ N(0, I)  +  y (rótulo)  →  ConditionalDecoder  →  imagem da classe y
```

> **O que observar:** cada linha corresponde a uma classe. As imagens da mesma linha devem ter características visuais em comum (forma, cor, textura). Diversidade entre linhas confirma que o condicionamento está funcionando.

In [ ]:
N_PER_CLASS = 4
cvae_model.eval()

fig, axes = plt.subplots(NUM_CLASSES, N_PER_CLASS,
                          figsize=(N_PER_CLASS * 2, NUM_CLASSES * 2))

with torch.no_grad():
    for class_idx in range(NUM_CLASSES):
        y = torch.full((N_PER_CLASS,), class_idx, dtype=torch.long, device=DEVICE)
        imgs = denormalize(cvae_model.generate(y))
        for col in range(N_PER_CLASS):
            axes[class_idx, col].imshow(imgs[col].permute(1, 2, 0).cpu().numpy())
            axes[class_idx, col].axis("off")
        axes[class_idx, 0].set_ylabel(IDX_TO_CLASS[class_idx],
                                       fontsize=9, rotation=0,
                                       labelpad=50, va="center")

plt.suptitle("cVAE — conditional generation (all classes, z ~ N(0, I))", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "cvae_generated_all_classes.png"), dpi=150, bbox_inches="tight")
with mlflow.start_run(run_id=cvae_run.info.run_id):
    mlflow.log_artifact(os.path.join(OUTPUT_DIR, "cvae_generated_all_classes.png"))
plt.show()
_path = os.path.join(OUTPUT_DIR, "cvae_generated_all_classes.png")
print(f"Saved to {_path}")

## 15. Comparação do espaço latente — VAE vs cVAE

Codifica as mesmas imagens com ambos os modelos e projeta os vetores latentes em 2D via **t-SNE**.

| Modelo | Espaço latente esperado |
|---|---|
| **VAE** | Classes misturadas — sem organização por categoria |
| **cVAE** | Clusters separados por classe — o rótulo `y` estrutura o espaço latente |

> **Interpretação:** clusters bem definidos no cVAE confirmam que o condicionamento funcionou — o encoder aprendeu a agrupar representações da mesma classe.

In [ ]:
from sklearn.manifold import TSNE
CLS_TO_IDX    = CLASS_TO_IDX
IDX_TO_CLS    = IDX_TO_CLASS
tsne_transform = eval_transform

TSNE_PER_CLASS = 150  # imagens por classe — menor = mais rápido

# Coleta caminhos de imagens e rótulos
tsne_paths, tsne_labels = [], []
for cls, idx in sorted(CLS_TO_IDX.items(), key=lambda x: x[1]):
    cls_dir = os.path.join(DATA_DIR, cls)
    files = [f for f in os.listdir(cls_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    for f in random.sample(files, min(TSNE_PER_CLASS, len(files))):
        tsne_paths.append(os.path.join(cls_dir, f))
        tsne_labels.append(idx)

tsne_labels_arr  = np.array(tsne_labels)
tsne_class_names = [IDX_TO_CLS[i] for i in range(NUM_CLASSES)]


def encode_for_tsne(enc_model, paths, labels, is_cvae, batch_size=128):
    enc_model.eval()
    mus = []
    for i in range(0, len(paths), batch_size):
        imgs = torch.stack([
            tsne_transform(Image.open(p).convert("RGB"))
            for p in paths[i : i + batch_size]
        ]).to(DEVICE)
        y = torch.tensor(labels[i : i + batch_size], dtype=torch.long, device=DEVICE)
        with torch.no_grad():
            mu, _ = enc_model.encoder(imgs, y) if is_cvae else enc_model.encoder(imgs)
        mus.append(mu.cpu().numpy())
    return np.concatenate(mus)


print(f"Codificando {len(tsne_paths)} imagens com VAE e cVAE...")
mus_vae  = encode_for_tsne(model,      tsne_paths, tsne_labels, is_cvae=False)
mus_cvae = encode_for_tsne(cvae_model, tsne_paths, tsne_labels, is_cvae=True)

print("Executando t-SNE × 2 (pode levar alguns minutos)...")
z_vae  = TSNE(n_components=2, perplexity=40, random_state=42, max_iter=1000).fit_transform(mus_vae)
z_cvae = TSNE(n_components=2, perplexity=40, random_state=42, max_iter=1000).fit_transform(mus_cvae)

colors = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, z2d, title in zip(axes, [z_vae, z_cvae], ["VAE", "cVAE"]):
    for idx, (name, color) in enumerate(zip(tsne_class_names, colors)):
        mask = tsne_labels_arr == idx
        ax.scatter(z2d[mask, 0], z2d[mask, 1], c=[color], label=name,
                   alpha=0.6, s=10, linewidths=0)
    ax.set_title(f"{title} — t-SNE", fontsize=12)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.legend(markerscale=3, framealpha=0.8, fontsize=9)

plt.suptitle("Comparação do espaço latente: VAE vs cVAE", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "latent_space_comparison.png"), dpi=150, bbox_inches="tight")
with mlflow.start_run(run_id=cvae_run.info.run_id):
    mlflow.log_artifact(os.path.join(OUTPUT_DIR, "latent_space_comparison.png"))
plt.show()
_path = os.path.join(OUTPUT_DIR, "latent_space_comparison.png")
print(f"Salvo em {_path}")